In [1]:
import pandas as pd
import json

def load_data(path, use_title=True):
	with open(path, encoding='utf-8') as f:
		raw = json.load(f)
	data = pd.DataFrame(raw['reviews'])
	
	star_ratings = {'1': 0, '2': 0, '4': 1, '5': 1}
	data['label'] = data['starRating'].map(star_ratings)

	data['title'] = data['title'].fillna('')
	data['content'] = data['content'].fillna('')

	if use_title:
		data['text'] = data['title'] + ' ' + data['content']
	else:
		data['text'] = data['content']

	data = data.drop(columns=['starRating', 'title', 'content'])
	return data

In [2]:
from preprocess import normalize, doc_to_tokens, load_nlp

nlp = load_nlp()

In [3]:
train = load_data('../data/train.json')
test = load_data('../data/test.json')

train.info()

<class 'pandas.DataFrame'>
RangeIndex: 12000 entries, 0 to 11999
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   index   12000 non-null  str  
 1   label   12000 non-null  int64
 2   text    12000 non-null  str  
dtypes: int64(1), str(2)
memory usage: 281.4 KB


In [4]:
train['n_words'] = train['text'].str.split().str.len()
print(train.columns, train.describe())

Index(['index', 'label', 'text', 'n_words'], dtype='str')               label       n_words
count  12000.000000  12000.000000
mean       0.500000     38.736417
std        0.500021     50.399980
min        0.000000      2.000000
25%        0.000000     11.000000
50%        0.500000     23.000000
75%        1.000000     47.000000
max        1.000000   1066.000000


In [5]:
train.groupby('label')['n_words'].describe()

,count,mean,std,min,25%,50%,75%,max
label,,,,,,,,
0,6000.0,43.571500,48.993744,2.0,15.0,29.0,54.0,1066.0
1,6000.0,33.901333,51.318501,2.0,8.0,18.0,39.0,798.0


In [6]:
train.head()

,index,label,text,n_words
0,11262,0,Foarte slab ca aspect este foarte frumoasa dar...,17
1,3890,1,Foarte multumit se incarca repede si tine 3 in...,12
2,9413,0,Țeapa de zile mari!!! in primul rand nu este d...,57
3,9350,0,Nu merita cumparate nu merita cumparate... sun...,33
4,7126,1,Recomand un ceas excelent. face cam tot ce ai ...,70


In [7]:
text = train['text'].str.cat(sep=' ')
print(len(text), text[:500])

2664965 Foarte slab ca aspect este foarte frumoasa dar cine vrea sa se joace cu tastatura nu poate Foarte multumit se incarca repede si tine 3 incarcari complete ale telefonului. Țeapa de zile mari!!! in primul rand nu este de stica dupa cum spune producatorul, este una de plastic de nici 10 lei   calitatea este 0 a mea a venit gata zgariata plus de asta nu se lipeste pe ecran sub nici o forma desi am urmat atent instructiunile   daca nu aveti pe ce da bani comandati! Nu merita cumparate nu merita cumpa


In [8]:
# are there any romanian characters?
from collections import Counter
import unicodedata

counts = Counter(text)

non_ascii = []
for char, count in counts.most_common():
	if char.isascii():
		continue
	non_ascii.append((char, count))

print(non_ascii[:40])
print(counts['ă'], counts['ș'], counts['ş'], counts['ț'], counts['ţ'])

# update: there aren't any diacritics / there are very few, negligible

[('”', 111), ('“', 84), ('̶', 84), ('„', 25), ('Î', 18), ('•', 14), ('️', 12), ('☺', 12), ('Ă', 9), ('Ț', 8), ('’', 6), ('…', 6), ('⛤', 6), ('❤', 5), ('⭐', 5), ('–', 5), ('Ș', 4), ('°', 4), ('€', 4), ('£', 3), ('×', 3), ('å', 3), ('✓', 3), ('↓', 2), ('í', 2), ('ä', 2), ('ƒ', 2), ('®', 2), ('—', 2), ('ω', 2), ('★', 2), ('İ', 1), ('̇', 1), ('à', 1), ('→', 1), ('é', 1), ('‘', 1), ('ű', 1), ('á', 1), ('✌', 1)]
0 0 0 0 0


In [9]:
# checking what method i should use for decomposition (stripping all diacritics)

test_char = 'Ț'
print(unicodedata.category(test_char))
test_char = unicodedata.normalize('NFD', test_char)
for c in test_char:
	print(unicodedata.category(c), c)

# confirmed: NFD

Lu
Lu T
Mn ̦


In [10]:
train['text_norm'] = train['text'].apply(normalize)
train['n_words_norm'] = train['text_norm'].str.split().str.len()

test['text_norm'] = test['text'].apply(normalize)
test['n_words_norm'] = test['text_norm'].str.split().str.len()

In [11]:
import spacy
nlp = spacy.load('ro_core_news_sm', enable=['tok2vec', 'lemmatizer'])

In [12]:
stop_words = nlp.Defaults.stop_words
print(len(stop_words))
stop_words_norm = {normalize(word) for word in stop_words}
print(stop_words_norm, len(stop_words_norm))

494
{'incotro', 'tai', 'noua', 'incit', 'rog', 'oricare', 'mata', 'fost', 'ei', 'oriunde', 'acel', 'neincetat', 'tale', 'sunteti', 'mine', 'cea', 'ar', 'atunci', 'avut', 'li', 'sase', 'ti', 'toti', 'cel', 'caut', 'atatea', 'din', 'intre', 'mea', 'timp', 'ala', 'de', 'alt', 'stiu', 'acord', 'am', 'catre', 'noi', 'fiti', 'ale', 'aiba', 'a', 'te', 'nu', 'spate', 'tot', 'acela', 'ce', 'nicidecum', 'cati', 'era', 'data', 'ati', 'desi', 'ta', 'apai', 'deunaseara', 'alea', 'pic', 'sa-mi', 'vostri', 'asta', 'r', 'acesta', 'altfel', 'vreme', 'noroc', 'acestui', 'conform', 'are', 'dintr', 'ma', 'putini', 'una', 't', 'atitia', 'toata', 'foarte', 'dupa', 'deasupra', 'aiurea', 'asa', 'fiu', 'oricat', 'ni', 'altceva', 'ii', 'prin', 'nimic', 'macar', 'daca', 'sub', 'sinteti', 'mi', 'care', 'alte', 'acea', 'putin', 'ceea', 'doar', 'intrucat', 'k', 'sintem', 'acolo', 'tine', 'zece', 'cineva', 'se', 'aceasta', 'despre', 'acum', 'inainte', 'inca', 'oricine', 'insa', 'pot', 'x', 'm', 'cum', 'unii', 'doi',

In [13]:
print(sorted(stop_words_norm))

['a', 'abia', 'acea', 'aceasta', 'aceea', 'aceeasi', 'acei', 'aceia', 'acel', 'acela', 'acelasi', 'acele', 'acelea', 'acest', 'acesta', 'aceste', 'acestea', 'acestei', 'acesti', 'acestia', 'acestui', 'acolo', 'acord', 'acum', 'adica', 'ai', 'aia', 'aiba', 'aici', 'aiurea', 'al', 'ala', 'alaturi', 'ale', 'alea', 'alt', 'alta', 'altceva', 'altcineva', 'alte', 'altfel', 'alti', 'altii', 'altul', 'am', 'anume', 'apai', 'apoi', 'ar', 'are', 'as', 'asa', 'asadar', 'asemenea', 'asta', 'astazi', 'astea', 'astfel', 'astia', 'asupra', 'atare', 'atat', 'atata', 'atatea', 'atatia', 'ati', 'atit', 'atita', 'atitea', 'atitia', 'atunci', 'au', 'avea', 'avem', 'aveti', 'avut', 'azi', 'b', 'ba', 'bine', 'bucur', 'buna', 'c', 'ca', 'caci', 'cam', 'cand', 'capat', 'care', 'carei', 'careia', 'caror', 'carora', 'carui', 'caruia', 'cat', 'cate', 'cati', 'catre', 'catva', 'caut', 'ce', 'cea', 'ceea', 'cei', 'ceilalti', 'cel', 'cele', 'celor', 'ceva', 'chiar', 'ci', 'cinci', 'cind', 'cine', 'cineva', 'cit', '

In [14]:
import re

rows = []
keep = {'aiurea', 'buna', 'bucur', 'contra', 'da', 'datorita', 'foarte', 'dar', 'insa', 'fara', 'nici', 'nimic', 'prea', 'bine', 'departe', 'frumos', 'impotriva', 'macar', 'mult', 'multa', 'multe', 'multi', 'multumesc', 'neincetat', 'nevoie', 'niciodata', 'noroc', 'nou', 'noua', 'nu', 'putin', 'putina', 'putini', 'zero'}

for word in sorted(keep | {'si'}):
	pattern = rf'\b{re.escape(word)}\b'
	has_word = train['text_norm'].str.contains(pattern, regex=True, case=False)
	freq = has_word.groupby(train['label']).mean()
	rows.append({
		'word': word,
		'neg': freq[0],
		'pos': freq[1],
		'count': has_word.sum(),
	})

stats = pd.DataFrame(rows)
stats['ratio'] = stats['neg'] / stats['pos']
stats = stats.sort_values('ratio', ascending=False)
stats

,word,neg,pos,count,ratio
20,neincetat,0.000167,0.000000,1,inf
14,macar,0.047000,0.005000,312,9.400000
34,zero,0.005000,0.000667,34,7.500000
22,nici,0.175833,0.037667,1281,4.668142
25,noroc,0.008000,0.001833,59,4.363636
26,nou,0.018667,0.005333,144,3.500000
0,aiurea,0.007667,0.002333,60,3.285714
24,nimic,0.056500,0.017667,445,3.198113
28,nu,0.852833,0.292500,6872,2.915670
8,departe,0.009500,0.003667,79,2.590909


In [15]:
# my experiments proved that the stop_words list from spaCy was full of important words, such as 'nu', 'macar' or 'zero'

custom_stop_words = stop_words_norm - keep
print(len(stop_words_norm) - len(custom_stop_words))
stats.to_csv('../results/caught_words.csv')

34


In [16]:
example = train[train['text'].str.contains('lei')]
example = example.iloc[58]['text_norm']
doc = nlp('s-a stricat intr-o luna | nu-mi place')

In [17]:
testing = []
for token in doc:
	testing.append({
		'token': token,
		'alpha': token.is_alpha,
		'lemma_': token.lemma_,
		'lemma': token.lemma,
		'space': token.is_space,
		'num': token.like_num,
		'punctuation': token.is_punct,
	})

df = pd.DataFrame(testing)
df

,token,alpha,lemma_,lemma,space,num,punctuation
0,s-,False,sine,17209644349998549839,False,False,False
1,a,True,avea,6634187297232829051,False,False,False
2,stricat,True,strica,2693495820325159610,False,False,False
3,intr-,False,intru,17339140353501564234,False,False,False
4,o,True,un,12188161997964598936,False,False,False
5,luna,True,lună,8140304746393508277,False,False,False
6,|,False,|,1965435164061188486,False,False,False
7,nu,True,nu,7171486237877634746,False,False,False
8,-mi,False,eu,4859553437162253712,False,False,False
9,place,True,plăcea,12022447512639822033,False,False,False


In [20]:
res = doc_to_tokens(doc, use_lemma=True, stop_words=custom_stop_words)
res

['sine', 'strica', 'intru', 'luna', 'nu', 'placea']

In [21]:
def process(df, nlp, batch_size=1000):
	tokens = []
	lemmas = []

	for doc in nlp.pipe(df['text_norm'], batch_size=batch_size):
		tokens.append(doc_to_tokens(doc, use_lemma=False))
		lemmas.append(doc_to_tokens(doc, use_lemma=True))

	df = df.copy()
	df['tokens'] = tokens
	df['lemmas'] = lemmas
	return df

In [22]:
train_proc = process(train, nlp)
test_proc = process(test, nlp)

train_proc.to_pickle('../data/train_proc_title.pkl')
test_proc.to_pickle('../data/test_proc_title.pkl')